# Multi-Layer Focused Analysis (Layers 9, 13, 17)

**Objetivo:** Verificar se os achados de encoding locality generalizam além da layer 13, rodando probes dirigidos, steering causal de gênero com benchmark ampliado (~240 itens), e controles norm-matched em 3 layers.

**Layers:** 9 (early-mid), 13 (mid — análise principal), 17 (mid-late)

**Justificativa:** Jing et al. encontram sintaxe complexa em layers 3-8 e pragmática em 8-15 (modelo 32L). Layers 9-17 cobrem a zona equivalente no Gemma 2 2B (26L).

**Pipeline por layer:**
1. Probes dirigidos (6 fenômenos × 10 pares)
2. Análise de localidade com thresholds pré-fixados
3. Steering de gênero (top-3 candidates + fallback)
4. Controles norm-matched (5 features × 20 gerações)
5. Benchmark de gênero ampliado (~240 itens)

**Tempo estimado:** ~2.5h no T4/L4

In [ ]:
!pip install transformer_lens sae_lens scipy -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "./data/checkpoints/"
import os; os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
import torch, numpy as np, json, time, re, gc
from tqdm.auto import tqdm
from scipy.stats import fisher_exact

device = "cuda" if torch.cuda.is_available() else "cpu"
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
LAYERS = [9, 13, 17]

# --- Thresholds de regime PRÉ-FIXADOS (congelados antes de rodar) ---
# Baseados nos resultados existentes: gênero ~53%, crase ~23%, clíticos ~33%
REGIME_THRESHOLDS = {
    "localized": 0.45,       # conc_top1_top5 >= 0.45
    "semi_localized": 0.25,  # 0.25 <= conc_top1_top5 < 0.45
    # distributed: conc_top1_top5 < 0.25
}

def classify_regime(conc):
    if conc >= REGIME_THRESHOLDS["localized"]:
        return "localized"
    elif conc >= REGIME_THRESHOLDS["semi_localized"]:
        return "semi-localized"
    return "distributed"

# --- Steering config ---
STEERING_PROMPTS = [
    "A menina bonita foi à",
    "A diretora da empresa apresentou a nova",
    "A professora explicou que a",
    "A médica disse que a paciente estava",
]
SEEDS = [42, 123, 456, 789, 1337]
N_CONTROLS = 5
MAX_NEW_TOKENS = 50
FALLBACK_THRESHOLD = 5  # min erros na top-1 para não testar fallback

GENDER_ERROR_PATTERNS = [
    r'\b[Aa]\s+(?:menino|garoto|homem|pai|filho|marido|irmão|avô|tio|primo|sobrinho|neto|bolo|carro|livro|gato|cachorro|apartamento|prédio|rio|muro|banheiro|comércio|veículo|sistema|tratamento|avanço|consumo|câncer)\b',
    r'\b[Oo]\s+(?:menina|garota|mulher|mãe|filha|esposa|irmã|avó|tia|prima|sobrinha|neta|casa|mesa|porta|cadeira|escola|cidade|rua|água|comida|roupa|praia|música|festa|carta)\b',
    r'\b[Uu]ma?\s+(?:menino|garoto|pai|filho|bolo|gato|carro)\b',
    r'\b[Uu]m\s+(?:menina|garota|mãe|filha|casa|mesa|escola)\b',
    r'\b(?:bonita|dedicada|nova|velha|alta|baixa|gorda|magra|loira|morena)\s+(?:menino|garoto|homem|pai|filho|professor|diretor|médico|engenheiro|aluno)\b',
    r'\b(?:bonito|dedicado|novo|velho|alto|baixo|gordo|magro|loiro|moreno)\s+(?:menina|garota|mulher|mãe|filha|professora|diretora|médica|engenheira|aluna)\b',
]

print(f"Config: layers={LAYERS}, seeds={SEEDS}, prompts={len(STEERING_PROMPTS)}, controls={N_CONTROLS}")
print(f"Regime thresholds: {REGIME_THRESHOLDS}")

In [ ]:
from sae_lens import SAE, HookedSAETransformer
print("Carregando modelo...")
model = HookedSAETransformer.from_pretrained_no_processing("gemma-2-2b", device=device, dtype=torch.float16)
tokenizer = model.tokenizer
print(f"Modelo carregado. Layers: {model.cfg.n_layers}, d_model: {model.cfg.d_model}")

## 1. Probes dirigidos: 6 fenômenos × 10 pares mínimos

In [ ]:
PROBES = {
    "gender": {
        "positive": [
            "A menina bonita correu pelo parque.",
            "A professora dedicada ensinou a lição.",
            "A diretora nova chegou na escola.",
            "A médica experiente atendeu o paciente.",
            "A engenheira brasileira projetou a ponte.",
            "A menina inteligente resolveu o problema.",
            "A advogada famosa venceu o caso.",
            "A cantora talentosa encantou a plateia.",
            "A cozinheira habilidosa preparou o jantar.",
            "A estudante aplicada passou no exame.",
        ],
        "negative": [
            "O menino bonito correu pelo parque.",
            "O professor dedicado ensinou a lição.",
            "O diretor novo chegou na escola.",
            "O médico experiente atendeu o paciente.",
            "O engenheiro brasileiro projetou a ponte.",
            "O menino inteligente resolveu o problema.",
            "O advogado famoso venceu o caso.",
            "O cantor talentoso encantou a plateia.",
            "O cozinheiro habilidoso preparou o jantar.",
            "O estudante aplicado passou no exame.",
        ],
    },
    "crase": {
        "positive": [
            "Ele foi à escola todos os dias.",
            "Ela se referiu à diretora do colégio.",
            "O aluno entregou o trabalho à professora.",
            "Nós fomos à praia no fim de semana.",
            "Ele assistiu à palestra com atenção.",
            "Ela voltou à cidade natal depois de anos.",
            "O diretor respondeu à carta do ministro.",
            "Nós nos dirigimos à saída de emergência.",
            "Ela se dedicou à pesquisa por décadas.",
            "O turista foi à catedral mais antiga.",
        ],
        "negative": [
            "Ele foi ao mercado todos os dias.",
            "Ela se referiu ao diretor do colégio.",
            "O aluno entregou o trabalho ao professor.",
            "Nós fomos ao parque no fim de semana.",
            "Ele assistiu ao jogo com atenção.",
            "Ele voltou ao escritório depois de anos.",
            "O diretor respondeu ao ofício do ministro.",
            "Nós nos dirigimos ao portão de emergência.",
            "Ele se dedicou ao estudo por décadas.",
            "O turista foi ao museu mais antigo.",
        ],
    },
    "clitics": {
        "positive": [
            "Ele me disse a verdade ontem.",
            "O professor nos explicou a matéria.",
            "Ela se arrumou para a festa.",
            "Ele lhe entregou o presente.",
            "Eles nos convidaram para o jantar.",
            "A mãe me chamou para almoçar.",
            "O chefe nos pediu para ficar.",
            "Ela se preparou cuidadosamente.",
            "Ele lhe prometeu que voltaria.",
            "Eles nos avisaram do perigo.",
        ],
        "negative": [
            "He told me the truth yesterday.",
            "The teacher explained the subject to us.",
            "She got ready for the party.",
            "He gave her the present.",
            "They invited us for dinner.",
            "Mom called me for lunch.",
            "The boss asked us to stay.",
            "She prepared herself carefully.",
            "He promised her he would return.",
            "They warned us of the danger.",
        ],
    },
    "verbal_conjugation": {
        "positive": [
            "Nós cantamos a música juntos ontem.",
            "Eles correram pela rua toda manhã.",
            "Eu falei com o diretor sobre isso.",
            "Vocês terminaram o trabalho a tempo.",
            "Nós estudávamos quando a luz apagou.",
            "Eles fizeram tudo que foi pedido.",
            "Eu escrevi a carta para ela.",
            "Vocês compreenderam o exercício completo.",
            "Nós partiremos amanhã bem cedo.",
            "Eles trouxeram os documentos necessários.",
        ],
        "negative": [
            "We sang the song together yesterday.",
            "They ran down the street every morning.",
            "I spoke with the director about this.",
            "You finished the work on time.",
            "We were studying when the light went out.",
            "They did everything that was asked.",
            "I wrote the letter for her.",
            "You understood the complete exercise.",
            "We will leave tomorrow very early.",
            "They brought the necessary documents.",
        ],
    },
    "personal_infinitive": {
        "positive": [
            "É importante nós estudarmos para a prova.",
            "Seria bom vocês chegarem mais cedo.",
            "Para eles entenderem, é preciso explicar.",
            "Antes de nós sairmos, tranque a porta.",
            "É fundamental os alunos participarem.",
            "Basta vocês assinarem o documento.",
            "Convém nós analisarmos os dados antes.",
            "É hora de eles decidirem o que fazer.",
            "Para nós avançarmos, precisamos de apoio.",
            "É essencial vocês contribuírem com ideias.",
        ],
        "negative": [
            "É importante estudar para a prova.",
            "Seria bom chegar mais cedo.",
            "Para entender, é preciso explicar.",
            "Antes de sair, tranque a porta.",
            "É fundamental participar da reunião.",
            "Basta assinar o documento corretamente.",
            "Convém analisar os dados com calma.",
            "É hora de decidir o que fazer.",
            "Para avançar, precisamos de apoio total.",
            "É essencial contribuir com boas ideias.",
        ],
    },
    "nominal_agreement": {
        "positive": [
            "As meninas bonitas correram pelo parque.",
            "Os livros antigos ficaram na estante.",
            "As casas grandes foram pintadas de azul.",
            "Os professores dedicados ensinaram bem.",
            "As flores vermelhas enfeitavam o jardim.",
            "Os carros novos estacionaram na garagem.",
            "As crianças pequenas brincavam no quintal.",
            "Os alunos aplicados foram premiados.",
            "As portas abertas ventilavam a sala.",
            "Os gatos pretos dormiram no sofá.",
        ],
        "negative": [
            "A menina bonita correu pelo parque.",
            "O livro antigo ficou na estante.",
            "A casa grande foi pintada de azul.",
            "O professor dedicado ensinou bem.",
            "A flor vermelha enfeitava o jardim.",
            "O carro novo estacionou na garagem.",
            "A criança pequena brincava no quintal.",
            "O aluno aplicado foi premiado.",
            "A porta aberta ventilava a sala.",
            "O gato preto dormiu no sofá.",
        ],
    },
}

print(f"Total de fenômenos: {len(PROBES)}")
for p, v in PROBES.items():
    print(f"  {p}: {len(v['positive'])} pares")

## 2. Benchmark de gênero ampliado (~240 itens)

40 adjetivos biformes × 3 templates × 2 gêneros = 240 itens.
Filtro: remove invariáveis (correto == incorreto).

In [ ]:
ADJECTIVES = [
    ("bonito", "bonita"), ("alto", "alta"), ("baixo", "baixa"),
    ("gordo", "gorda"), ("magro", "magra"), ("velho", "velha"),
    ("novo", "nova"), ("feio", "feia"), ("rico", "rica"),
    ("pobre", "pobre"), ("cansado", "cansada"), ("animado", "animada"),
    ("dedicado", "dedicada"), ("organizado", "organizada"),
    ("preocupado", "preocupada"), ("calmo", "calma"),
    ("nervoso", "nervosa"), ("famoso", "famosa"),
    ("talentoso", "talentosa"), ("corajoso", "corajosa"),
    ("preguiçoso", "preguiçosa"), ("cuidadoso", "cuidadosa"),
    ("carinhoso", "carinhosa"), ("generoso", "generosa"),
    ("sério", "séria"), ("simpático", "simpática"),
    ("antipático", "antipática"), ("tímido", "tímida"),
    ("atrevido", "atrevida"), ("educado", "educada"),
    ("esperto", "esperta"), ("lento", "lenta"),
    ("rápido", "rápida"), ("quieto", "quieta"),
    ("barulhento", "barulhenta"), ("chato", "chata"),
    ("divertido", "divertida"), ("salgado", "salgada"),
    ("amargo", "amarga"), ("doce", "doce"),
]

SUBJECTS_FEM = [
    "A menina", "A professora", "A médica", "A engenheira", "A advogada",
    "A diretora", "A aluna", "A cantora", "A cozinheira", "A vizinha",
    "A mãe", "A filha", "A irmã", "A avó", "A tia",
    "A enfermeira", "A motorista", "A secretária", "A escritora", "A atriz",
]

SUBJECTS_MASC = [
    "O menino", "O professor", "O médico", "O engenheiro", "O advogado",
    "O diretor", "O aluno", "O cantor", "O cozinheiro", "O vizinho",
    "O pai", "O filho", "O irmão", "O avô", "O tio",
    "O enfermeiro", "O motorista", "O secretário", "O escritor", "O ator",
]

TEMPLATES = [
    "{subj} é muito",
    "{subj} parece",
    "{subj} ficou muito",
]

import random
random.seed(42)

GENDER_BENCHMARK = []
for adj_m, adj_f in ADJECTIVES:
    if adj_m == adj_f:
        continue
    for template in TEMPLATES:
        subj_f = random.choice(SUBJECTS_FEM)
        subj_m = random.choice(SUBJECTS_MASC)
        GENDER_BENCHMARK.append((template.format(subj=subj_f), adj_f, adj_m, "F"))
        GENDER_BENCHMARK.append((template.format(subj=subj_m), adj_m, adj_f, "M"))

random.shuffle(GENDER_BENCHMARK)

n_fem = sum(1 for _, _, _, g in GENDER_BENCHMARK if g == "F")
n_masc = sum(1 for _, _, _, g in GENDER_BENCHMARK if g == "M")
print(f"Benchmark total: {len(GENDER_BENCHMARK)} itens ({n_fem} F, {n_masc} M)")
print(f"Exemplos:")
for ctx, corr, incorr, g in GENDER_BENCHMARK[:5]:
    print(f"  [{g}] {ctx} → {corr} (vs {incorr})")

## 3. Funções auxiliares

In [ ]:
def get_feature_activations(model, sae, tokenizer, text, hook_name):
    input_ids = tokenizer.encode(text, return_tensors="pt").to(device)
    with torch.no_grad():
        _, cache = model.run_with_cache(input_ids, names_filter=[hook_name])
        resid = cache[hook_name]
        acts = sae.encode(resid)
        mean_acts = acts.mean(dim=1).squeeze(0)
    del cache, resid, acts
    torch.cuda.empty_cache()
    return mean_acts.cpu()


def find_discriminative_features(model, sae, tokenizer, positive_texts, negative_texts,
                                  hook_name, top_k=20):
    pos_acts = torch.stack([get_feature_activations(model, sae, tokenizer, t, hook_name)
                            for t in positive_texts])
    neg_acts = torch.stack([get_feature_activations(model, sae, tokenizer, t, hook_name)
                            for t in negative_texts])

    mean_pos = pos_acts.mean(dim=0)
    mean_neg = neg_acts.mean(dim=0)
    std_pos = pos_acts.std(dim=0)
    std_neg = neg_acts.std(dim=0)

    diff = mean_pos - mean_neg
    pooled_std = torch.sqrt((std_pos**2 + std_neg**2) / 2 + 1e-8)
    effect_size = diff / pooled_std

    top_diff_idx = diff.abs().argsort(descending=True)[:top_k]

    return {
        "top_features": [(idx.item(), diff[idx].item(), effect_size[idx].item(),
                          mean_pos[idx].item(), mean_neg[idx].item())
                         for idx in top_diff_idx],
        "max_diff": diff.abs().max().item(),
        "max_effect_size": effect_size[top_diff_idx[0]].item(),
    }


def make_ablation_hook(sae, feature_id):
    fid_t = torch.tensor([feature_id], device=device)
    def hook_fn(value, hook):
        with torch.no_grad():
            sae_acts = sae.encode(value)
            modified = sae_acts.clone()
            modified[:, :, fid_t] = 0.0
            reconstructed = sae.decode(modified)
            error = value - sae.decode(sae_acts)
            return reconstructed + error
    return hook_fn


def generate_text(model, tokenizer, prompt, hook_name=None, hook_fn=None,
                  max_new=MAX_NEW_TOKENS, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    for _ in range(max_new):
        with torch.no_grad():
            if hook_fn and hook_name:
                logits = model.run_with_hooks(generated, fwd_hooks=[(hook_name, hook_fn)])
            else:
                logits = model(generated)
            next_logits = logits[0, -1, :] / 0.7
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            if next_token.item() == tokenizer.eos_token_id:
                break
            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=-1)
    return tokenizer.decode(generated[0], skip_special_tokens=True)


def detect_gender_errors(text):
    matches = []
    for pattern in GENDER_ERROR_PATTERNS:
        found = re.findall(pattern, text)
        matches.extend(found)
    return matches


def get_token_prob(probs, tokenizer, word):
    ids = tokenizer.encode(" " + word, add_special_tokens=False)
    ids_no_space = tokenizer.encode(word, add_special_tokens=False)
    all_ids = list(set(ids + ids_no_space))
    return sum(probs[tid].item() for tid in all_ids)


def evaluate_benchmark(model, tokenizer, benchmark, hook_name=None, hook_fn=None):
    results = []
    for ctx, correct_adj, incorrect_adj, gender in benchmark:
        input_ids = tokenizer.encode(ctx, return_tensors="pt").to(device)
        with torch.no_grad():
            if hook_fn and hook_name:
                logits = model.run_with_hooks(input_ids, fwd_hooks=[(hook_name, hook_fn)])
            else:
                logits = model(input_ids)
        probs = torch.softmax(logits[0, -1, :].float(), dim=-1)
        p_correct = get_token_prob(probs, tokenizer, correct_adj)
        p_incorrect = get_token_prob(probs, tokenizer, incorrect_adj)
        results.append({
            "context": ctx, "correct": correct_adj, "incorrect": incorrect_adj,
            "gender": gender, "p_correct": p_correct, "p_incorrect": p_incorrect,
            "is_correct": p_correct > p_incorrect,
        })
    return results


print("Funções carregadas.")

## 4. Executar probes em cada layer

In [ ]:
all_layer_results = {}
sae_cache = {}

for layer in LAYERS:
    hook_name = f"blocks.{layer}.hook_resid_post"
    sae_id = f"layer_{layer}/width_16k/canonical"

    print(f"\n{'='*60}")
    print(f"LAYER {layer}")
    print(f"{'='*60}")

    print(f"Carregando SAE layer {layer}...")
    sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=sae_id, device=device)
    sae_cache[layer] = sae

    layer_results = {}
    for phenomenon, texts in PROBES.items():
        print(f"  Probing: {phenomenon}...")
        t0 = time.time()
        result = find_discriminative_features(
            model, sae, tokenizer,
            texts["positive"], texts["negative"],
            hook_name, top_k=20
        )
        elapsed = time.time() - t0
        layer_results[phenomenon] = result

        top3 = result["top_features"][:3]
        print(f"    Top-3: {[(f, f'{d:+.3f}', f'd={e:.1f}') for f, d, e, _, _ in top3]} ({elapsed:.1f}s)")

    all_layer_results[layer] = layer_results

print("\nProbes concluídos para todas as layers.")

## 5. Análise de localidade com thresholds pré-fixados

In [ ]:
print("="*80)
print(f"ANÁLISE DE LOCALIDADE (thresholds: localized>={REGIME_THRESHOLDS['localized']}, semi>={REGIME_THRESHOLDS['semi_localized']})")
print("="*80)
print(f"\n{'Fenômeno':<25} {'Layer':>6} {'Top1/Top5':>12} {'Regime':>18}")
print("-"*65)

locality_analysis = {}
for phenomenon in PROBES.keys():
    locality_analysis[phenomenon] = {}
    for layer in LAYERS:
        result = all_layer_results[layer][phenomenon]
        diffs = [abs(d) for _, d, _, _, _ in result["top_features"][:20]]
        top1 = diffs[0]
        top5 = sum(diffs[:5])
        top20 = sum(diffs[:20])
        c1_5 = top1 / (top5 + 1e-8)
        regime = classify_regime(c1_5)

        locality_analysis[phenomenon][layer] = {
            "top1": top1, "top5": top5, "top20": top20,
            "conc_1_5": c1_5, "regime": regime,
        }
        print(f"{phenomenon:<25} {layer:>6} {c1_5:>11.1%} {regime:>18}")

print(f"\n{'='*80}")
print("MELHOR LAYER POR FENÔMENO (maior |diff|)")
print("="*80)
summary_table = {}
for phenomenon in PROBES.keys():
    summary_table[phenomenon] = {}
    for layer in LAYERS:
        r = all_layer_results[layer][phenomenon]
        summary_table[phenomenon][layer] = {
            "max_diff": r["max_diff"], "effect_size": r["max_effect_size"]
        }
    best = max(LAYERS, key=lambda l: summary_table[phenomenon][l]["max_diff"])
    d = summary_table[phenomenon][best]["effect_size"]
    print(f"  {phenomenon:<25} → Layer {best} (d={d:.2f})")

## 6. Steering de gênero por layer (top-3 candidates + fallback)

In [ ]:
gender_steering_results = {}

for layer in LAYERS:
    print(f"\n{'='*60}")
    print(f"STEERING DE GÊNERO — LAYER {layer}")
    print(f"{'='*60}")

    hook_name = f"blocks.{layer}.hook_resid_post"
    sae = sae_cache[layer]
    gender_result = all_layer_results[layer]["gender"]

    candidates = []
    for fid, diff, es, mp, mn in gender_result["top_features"][:3]:
        candidates.append({"feature_id": fid, "diff": diff, "effect_size": es,
                           "mean_pos": mp, "mean_neg": mn})
    print(f"  Candidates: {[(c['feature_id'], f'd={c["effect_size"]:.2f}') for c in candidates]}")

    effective_feature = None
    fallback_used = False
    all_generations = []

    for rank, candidate in enumerate(candidates):
        fid = candidate["feature_id"]
        print(f"\n  Testando feature {fid} (rank {rank+1})...")
        hook_fn = make_ablation_hook(sae, fid)

        generations = []
        error_count = 0
        for prompt in STEERING_PROMPTS:
            for seed in SEEDS:
                baseline = generate_text(model, tokenizer, prompt, seed=seed)
                ablated = generate_text(model, tokenizer, prompt, hook_name, hook_fn, seed=seed)
                errors = detect_gender_errors(ablated)
                gen = {
                    "prompt": prompt, "seed": seed,
                    "baseline": baseline, "ablated": ablated,
                    "regex_hit": len(errors) > 0, "regex_matches": errors,
                    "manual_flag": None,
                }
                generations.append(gen)
                if errors:
                    error_count += 1

        print(f"    Erros detectados: {error_count}/20")

        if rank == 0:
            all_generations = generations
            if error_count >= FALLBACK_THRESHOLD:
                effective_feature = fid
                print(f"    ✓ Top-1 suficiente ({error_count} >= {FALLBACK_THRESHOLD})")
                break
            else:
                print(f"    ✗ Top-1 insuficiente ({error_count} < {FALLBACK_THRESHOLD}), testando fallback...")
                fallback_used = True
        else:
            if error_count > len(all_generations) and error_count >= FALLBACK_THRESHOLD:
                all_generations = generations
                effective_feature = fid
                print(f"    ✓ Fallback rank {rank+1} aceito ({error_count} erros)")
                break
            elif rank == 2:
                best_rank = max(range(3), key=lambda r: sum(1 for g in (generations if r == rank else all_generations) if g["regex_hit"]))
                effective_feature = candidates[best_rank]["feature_id"] if best_rank == 0 else fid
                print(f"    Usando melhor disponível: feature {effective_feature}")

    if effective_feature is None:
        effective_feature = candidates[0]["feature_id"]

    gender_steering_results[layer] = {
        "candidates": candidates,
        "effective_feature": effective_feature,
        "fallback_used": fallback_used,
        "generations": all_generations,
        "error_count": sum(1 for g in all_generations if g["regex_hit"]),
    }
    print(f"\n  → Feature efetiva layer {layer}: {effective_feature} (fallback={fallback_used})")

print("\n" + "="*60)
print("RESUMO STEERING")
for layer in LAYERS:
    r = gender_steering_results[layer]
    print(f"  Layer {layer}: feature {r['effective_feature']}, erros {r['error_count']}/20, fallback={r['fallback_used']}")

## 7. Controles norm-matched por layer (5 × 20 gerações)

In [ ]:
control_results = {}

for layer in LAYERS:
    print(f"\n{'='*60}")
    print(f"CONTROLES NORM-MATCHED — LAYER {layer}")
    print(f"{'='*60}")

    hook_name = f"blocks.{layer}.hook_resid_post"
    sae = sae_cache[layer]
    eff_fid = gender_steering_results[layer]["effective_feature"]

    target_norm = sae.W_dec.data[eff_fid].norm().item()
    all_norms = sae.W_dec.data.norm(dim=1)
    norm_diffs = (all_norms - target_norm).abs()
    norm_diffs[eff_fid] = float('inf')
    candidates = norm_diffs.argsort()[:N_CONTROLS].tolist()

    print(f"  Target feature: {eff_fid} (norm={target_norm:.4f})")
    print(f"  Control features: {candidates}")
    print(f"  Control norms: {[f'{all_norms[c]:.4f}' for c in candidates]}")

    layer_controls = []
    for ctrl_fid in candidates:
        hook_fn = make_ablation_hook(sae, ctrl_fid)
        ctrl_errors = 0
        for prompt in STEERING_PROMPTS:
            for seed in SEEDS:
                text = generate_text(model, tokenizer, prompt, hook_name, hook_fn, seed=seed)
                errors = detect_gender_errors(text)
                if errors:
                    ctrl_errors += 1
        layer_controls.append({"feature_id": ctrl_fid, "errors_20": ctrl_errors,
                               "norm": all_norms[ctrl_fid].item()})
        print(f"    Control {ctrl_fid}: {ctrl_errors}/20 erros")

    control_results[layer] = {
        "target_feature": eff_fid,
        "target_norm": target_norm,
        "controls": layer_controls,
        "total_control_errors": sum(c["errors_20"] for c in layer_controls),
        "total_control_generations": N_CONTROLS * 20,
    }
    total_err = control_results[layer]["total_control_errors"]
    total_gen = control_results[layer]["total_control_generations"]
    print(f"\n  → Total controles layer {layer}: {total_err}/{total_gen} erros")

print("\nControles concluídos.")

## 8. Benchmark de gênero ampliado por layer

In [ ]:
benchmark_results = {}

for layer in LAYERS:
    print(f"\n{'='*60}")
    print(f"BENCHMARK DE GÊNERO — LAYER {layer}")
    print(f"{'='*60}")

    hook_name = f"blocks.{layer}.hook_resid_post"
    sae = sae_cache[layer]
    eff_fid = gender_steering_results[layer]["effective_feature"]
    hook_fn = make_ablation_hook(sae, eff_fid)

    print(f"  Feature: {eff_fid}")
    print(f"  Avaliando baseline...")
    baseline = evaluate_benchmark(model, tokenizer, GENDER_BENCHMARK)

    print(f"  Avaliando ablated...")
    ablated = evaluate_benchmark(model, tokenizer, GENDER_BENCHMARK, hook_name, hook_fn)

    ctrl_accs = []
    for i, ctrl in enumerate(control_results[layer]["controls"]):
        ctrl_hook = make_ablation_hook(sae, ctrl["feature_id"])
        ctrl_res = evaluate_benchmark(model, tokenizer, GENDER_BENCHMARK, hook_name, ctrl_hook)
        ctrl_acc = sum(1 for r in ctrl_res if r["is_correct"]) / len(ctrl_res)
        ctrl_accs.append(ctrl_acc)
        print(f"    Control {i+1}: {ctrl_acc:.1%}")

    def compute_stats(results):
        fem = [r for r in results if r["gender"] == "F"]
        masc = [r for r in results if r["gender"] == "M"]
        return {
            "overall": sum(1 for r in results if r["is_correct"]) / len(results),
            "feminine": sum(1 for r in fem if r["is_correct"]) / len(fem) if fem else 0,
            "masculine": sum(1 for r in masc if r["is_correct"]) / len(masc) if masc else 0,
            "n_fem": len(fem), "n_masc": len(masc), "n_total": len(results),
            "fem_correct": sum(1 for r in fem if r["is_correct"]),
            "masc_correct": sum(1 for r in masc if r["is_correct"]),
        }

    base_stats = compute_stats(baseline)
    abl_stats = compute_stats(ablated)

    fem_table = [[abl_stats["fem_correct"], base_stats["n_fem"] - abl_stats["fem_correct"]],
                 [base_stats["fem_correct"], base_stats["n_fem"] - base_stats["fem_correct"]]]
    _, fisher_p = fisher_exact(fem_table)

    benchmark_results[layer] = {
        "effective_feature": eff_fid,
        "baseline": base_stats,
        "ablated": abl_stats,
        "control_accs": ctrl_accs,
        "control_mean_acc": np.mean(ctrl_accs),
        "fisher_p_feminine": fisher_p,
        "fem_drop_pp": (base_stats["feminine"] - abl_stats["feminine"]) * 100,
        "masc_drop_pp": (base_stats["masculine"] - abl_stats["masculine"]) * 100,
    }

    print(f"\n  Baseline:  F={base_stats['feminine']:.1%} M={base_stats['masculine']:.1%} All={base_stats['overall']:.1%}")
    print(f"  Ablated:   F={abl_stats['feminine']:.1%} M={abl_stats['masculine']:.1%} All={abl_stats['overall']:.1%}")
    print(f"  Controls:  {[f'{a:.1%}' for a in ctrl_accs]}")
    print(f"  Fem drop:  {benchmark_results[layer]['fem_drop_pp']:+.1f}pp")
    print(f"  Fisher p (fem): {fisher_p:.4f}")

print("\nBenchmark concluído.")

## 9. Correspondência funcional cross-layer

In [ ]:
print("="*80)
print("CORRESPONDÊNCIA FUNCIONAL CROSS-LAYER")
print("="*80)

cross_layer = {}
for layer in LAYERS:
    eff = gender_steering_results[layer]["effective_feature"]
    errs = gender_steering_results[layer]["error_count"]
    regime = locality_analysis["gender"][layer]["regime"]
    fem_drop = benchmark_results[layer]["fem_drop_pp"]
    ctrl_errs = control_results[layer]["total_control_errors"]

    cross_layer[layer] = {
        "feature_id": eff,
        "steering_errors_20": errs,
        "control_errors": ctrl_errs,
        "regime": regime,
        "benchmark_fem_drop_pp": fem_drop,
        "same_effect_direction": fem_drop > 0,
    }

    status = "✓" if fem_drop > 0 and errs > 0 else "✗"
    print(f"  Layer {layer}: feature {eff}, regime={regime}, errs={errs}/20, " +
          f"ctrl={ctrl_errs}/{N_CONTROLS*20}, fem_drop={fem_drop:+.1f}pp {status}")

feature_ids = [cross_layer[l]["feature_id"] for l in LAYERS]
all_same_direction = all(cross_layer[l]["same_effect_direction"] for l in LAYERS)
all_same_regime = len(set(cross_layer[l]["regime"] for l in LAYERS)) == 1

print(f"\n  Feature IDs: {feature_ids}")
print(f"  Todas mesma direção de efeito: {all_same_direction}")
print(f"  Todas mesmo regime: {all_same_regime}")
if all_same_regime:
    print(f"  Regime comum: {cross_layer[LAYERS[0]]['regime']}")

## 10. Tabela comparativa final

In [ ]:
print("="*100)
print("TABELA COMPARATIVA: LOCALIDADE POR FENÔMENO × LAYER")
print("="*100)
print(f"{'Fenômeno':<25}", end="")
for l in LAYERS:
    print(f" {'Layer '+str(l):>20}", end="")
print()
print("-"*85)
for phenomenon in PROBES.keys():
    print(f"{phenomenon:<25}", end="")
    for l in LAYERS:
        la = locality_analysis[phenomenon][l]
        print(f" {la['conc_1_5']:.1%} ({la['regime'][:4]})".rjust(20), end="")
    print()

print(f"\n{'='*100}")
print("TABELA: BENCHMARK DE GÊNERO POR LAYER")
print("="*100)
print(f"{'Layer':>6} {'Feature':>8} {'Base F%':>10} {'Abl F%':>10} {'Drop pp':>10} {'Fisher p':>10} {'Ctrl errs':>12}")
print("-"*70)
for l in LAYERS:
    br = benchmark_results[l]
    cr = control_results[l]
    print(f"{l:>6} {br['effective_feature']:>8} {br['baseline']['feminine']:>9.1%} {br['ablated']['feminine']:>9.1%} "
          f"{br['fem_drop_pp']:>+9.1f} {br['fisher_p_feminine']:>9.4f} {cr['total_control_errors']:>5}/{cr['total_control_generations']}")

## 11. Salvar resultados

In [ ]:
results = {
    "experiment": "multilayer_focused",
    "model": "gemma-2-2b",
    "layers": LAYERS,
    "regime_thresholds": REGIME_THRESHOLDS,
    "n_benchmark_items": len(GENDER_BENCHMARK),
    "phenomena": list(PROBES.keys()),
    "probe_results": {},
    "locality_analysis": {},
    "gender_steering": {},
    "control_results": {},
    "benchmark_results": {},
    "cross_layer_correspondence": {},
}

for phenomenon in PROBES.keys():
    results["probe_results"][phenomenon] = {}
    results["locality_analysis"][phenomenon] = {}
    for layer in LAYERS:
        sl = str(layer)
        r = all_layer_results[layer][phenomenon]
        results["probe_results"][phenomenon][sl] = {
            "top_features": [
                {"feature_id": f, "diff": d, "effect_size": e, "mean_pos": mp, "mean_neg": mn}
                for f, d, e, mp, mn in r["top_features"]
            ],
            "max_diff": r["max_diff"],
            "max_effect_size": r["max_effect_size"],
        }
        results["locality_analysis"][phenomenon][sl] = locality_analysis[phenomenon][layer]

for layer in LAYERS:
    sl = str(layer)
    gs = gender_steering_results[layer]
    results["gender_steering"][sl] = {
        "candidates": gs["candidates"],
        "effective_feature": gs["effective_feature"],
        "fallback_used": gs["fallback_used"],
        "error_count": gs["error_count"],
        "generations": gs["generations"],
    }
    results["control_results"][sl] = control_results[layer]
    results["benchmark_results"][sl] = {
        k: v for k, v in benchmark_results[layer].items()
    }
    results["cross_layer_correspondence"][sl] = cross_layer[layer]

def make_serializable(obj):
    if isinstance(obj, (np.floating, np.integer)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_serializable(v) for v in obj]
    return obj

results = make_serializable(results)

out_path = os.path.join(SAVE_DIR, "exp_multilayer_focused_results.json")
with open(out_path, "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False, default=str)

local_path = "/content/exp_multilayer_focused_results.json"
with open(local_path, "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False, default=str)

print(f"Salvo em: {out_path}")
print(f"Salvo em: {local_path}")
print(f"\nJSON total: {len(json.dumps(results)):,} chars")
print("\nExperimento concluído!")

In [ ]:
for layer in list(sae_cache.keys()):
    del sae_cache[layer]
gc.collect()
torch.cuda.empty_cache()
print("Memória liberada.")